# 4 — Évaluation expérimentale et grille humaine

Le sujet demande une évaluation humaine d’au moins 60 QCM : clarté, difficulté et unicité de la bonne réponse. Cette cellule prépare une grille reproductible de 60 questions (ou moins si le corpus ne permet pas d’en produire 60). Deux évaluateurs remplissent leurs colonnes de façon indépendante. Aucun score humain ni kappa ne sera calculé avant la saisie effective des annotations.

In [1]:
from pathlib import Path
import sys, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'requirements.txt').exists(): ROOT = ROOT.parent
if not (ROOT / 'requirements.txt').exists(): raise FileNotFoundError('Racine du projet introuvable.')
sys.path.insert(0, str(ROOT))
from src.qcm_pipeline import cohen_kappa_from_review
PROCESSED = ROOT / 'data' / 'processed'
accepted_path = PROCESSED / 'qcm_acceptes.csv'
candidates_path = PROCESSED / 'qcm_candidats.csv'
if not candidates_path.exists(): raise FileNotFoundError('Exécuter les notebooks 01 et 02 avant l’évaluation.')
candidates = pd.read_csv(candidates_path)
validated_path = PROCESSED / 'qcm_valides_et_rejetes.csv'
if validated_path.exists():
    validator = pd.read_csv(validated_path)[['question_id', 'accepted']]
    candidates = candidates.merge(validator, on='question_id', how='left', validate='one_to_one')
else:
    candidates['accepted'] = False
candidates['accepted'] = candidates['accepted'].fillna(False).astype(bool)
accepted_pool = candidates[candidates['accepted']]
rejected_pool = candidates[~candidates['accepted']]
n_each = min(30, len(accepted_pool), len(rejected_pool))
parts = [accepted_pool.sample(n=n_each, random_state=SEED), rejected_pool.sample(n=n_each, random_state=SEED)]
selected_ids = set(pd.concat(parts)['question_id'])
remaining = candidates[~candidates['question_id'].isin(selected_ids)]
needed = min(60 - 2 * n_each, len(remaining))
if needed:
    parts.append(remaining.sample(n=needed, random_state=SEED))
review = pd.concat(parts).sample(frac=1, random_state=SEED).copy()
review['options'] = review['options'].map(lambda x: json.dumps(json.loads(x), ensure_ascii=False) if isinstance(x, str) else json.dumps(x, ensure_ascii=False))
review_path = PROCESSED / 'grille_evaluation_humaine.csv'
annotation_columns = [f'evaluateur_{e}_{field}' for e in [1, 2] for field in ['clarté', 'difficulté', 'unicité']]
annotation_columns += ['commentaire_evaluateur_1', 'commentaire_evaluateur_2']
if review_path.exists():
    previous = pd.read_csv(review_path)
    rating_columns = [f'evaluateur_{e}_{field}' for e in [1, 2] for field in ['clarté', 'difficulté', 'unicité']]
    complete_review = (len(previous) == 60 and previous['question_id'].is_unique
                       and all(column in previous.columns for column in rating_columns)
                       and previous[rating_columns].notna().all(axis=None))
    if complete_review:
        aligned = previous.merge(candidates[['question_id', 'question', 'accepted']],
                                 on='question_id', how='left', suffixes=('_fichier', '_projet'), validate='one_to_one')
        if aligned['accepted_projet'].isna().any():
            raise ValueError('Des questions évaluées ne figurent pas dans les candidats du projet.')
        if not (aligned['question_fichier'] == aligned['question_projet']).all():
            raise ValueError('Les questions évaluées ne correspondent pas aux QCM actuels.')
        if not (aligned['accepted_fichier'].astype(bool) == aligned['accepted_projet'].astype(bool)).all():
            raise ValueError('Les étiquettes acceptées/rejetées ne correspondent pas aux résultats QA.')
        review = previous.copy()
    else:
        keep = [column for column in annotation_columns if column in previous.columns]
        review = review.merge(previous[['question_id'] + keep], on='question_id', how='left', validate='one_to_one')
for column in annotation_columns:
    if column not in review.columns:
        review[column] = pd.NA if column.startswith('evaluateur_') else ''
    elif column.startswith('commentaire_'):
        review[column] = review[column].fillna('')
review.to_csv(review_path, index=False, encoding='utf-8-sig')
review = pd.read_csv(review_path)
print(f"Grille disponible : {review_path}")
print(f"Questions préparées : {len(review)} / 60")
display(review[['question_id', 'question', 'options', 'source_doc', 'source_page']].head(8))
print('Répartition acceptés/rejetés :')
display(review['accepted'].value_counts().rename(index={True: 'retenu par QA', False: 'rejeté par QA'}).to_frame('questions'))
print('Échelle recommandée : 1 = très faible ; 5 = très bon. Renseigner les colonnes evaluateur_1_* et evaluateur_2_* indépendamment.')

Grille disponible : D:\Generateur_QCM\data\processed\grille_evaluation_humaine.csv
Questions préparées : 60 / 60


C:\Users\user\AppData\Local\Temp\ipykernel_12272\3650121545.py:64: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  review[column] = review[column].fillna('')


,question_id,question,options,source_doc,source_page
0,Q0001,"Selon le support, quelle affirmation concernan...","[""1.Le cerveau devine d’abord la couleur de la...",01_vision_ordinateur_ch1.pdf,4
1,Q0003,"Selon le support, quelle affirmation concernan...","[""1.Le cerveau devine d’abord la couleur de la...",01_vision_ordinateur_ch1.pdf,4
2,Q0006,"Selon le support, quelle affirmation concernan...","[""Chaque élément de l'image est appelé « pixel...",01_vision_ordinateur_ch1.pdf,8
3,Q0007,"Selon le support, quelle affirmation concernan...","[""Les anciens systèmes étaient lents car ils d...",01_vision_ordinateur_ch1.pdf,9
4,Q0008,"Selon le support, quelle affirmation concernan...","[""L'algorithme traite les pixels, les structur...",01_vision_ordinateur_ch1.pdf,10
5,Q0010,"Selon le support, quelle affirmation concernan...","[""Suppression du flou Suppression du bruit Col...",01_vision_ordinateur_ch1.pdf,11
6,Q0011,"Selon le support, quelle affirmation concernan...","[""Réglage de la luminosité Amélioration du con...",01_vision_ordinateur_ch1.pdf,12
7,Q0013,"Selon le support, quelle affirmation concernan...","[""Détection de retouches et de montages -Détec...",01_vision_ordinateur_ch1.pdf,13


Répartition acceptés/rejetés :


,questions
accepted,
retenu par QA,53
rejeté par QA,7


Échelle recommandée : 1 = très faible ; 5 = très bon. Renseigner les colonnes evaluateur_1_* et evaluateur_2_* indépendamment.


## Résultats automatiques, taux de rejet et ablation

Le taux de rejet compare les candidats bruts aux QCM acceptés par QA. L’ablation ci-dessous résume aussi les contrôles d’ambiguïté et de question sans réponse. Une étude sur des contrôles construits ne remplace pas l’analyse humaine.

In [2]:
baseline_n = len(pd.read_csv(candidates_path))
advanced_n = int(candidates['accepted'].sum())
rejection_rate = 1 - advanced_n / baseline_n if baseline_n else float('nan')
gemini_validation_path = PROCESSED / 'qcm_valides_et_rejetes_gemini.csv'
gemini_validated = pd.read_csv(gemini_validation_path) if gemini_validation_path.exists() else pd.DataFrame()
summary_rows = [
    {'mesure': 'QCM générés — sans filtre (baseline)', 'valeur': baseline_n},
    {'mesure': 'QCM conservés — QA local TF-IDF', 'valeur': advanced_n},
    {'mesure': 'Taux de rejet QA local', 'valeur': rejection_rate},
    {'mesure': 'QCM dans la grille humaine', 'valeur': len(review)},
]
if not gemini_validated.empty:
    gemini_accepted_n = int(gemini_validated['accepted'].sum())
    gemini_rejection_rate = 1 - gemini_accepted_n / len(gemini_validated)
    summary_rows += [
        {'mesure': 'QCM conservés — QA Gemini', 'valeur': gemini_accepted_n},
        {'mesure': 'Taux de rejet QA Gemini', 'valeur': gemini_rejection_rate},
    ]
summary = pd.DataFrame(summary_rows)
display(summary)
control_path = ROOT / 'results' / 'resultats_controles_ablation.csv'
if control_path.exists():
    control_results = pd.read_csv(control_path)
    display(control_results)
else:
    print('Résultats d’ablation indisponibles : exécuter le notebook 03.')
human_ready = all(f'evaluateur_{e}_{field}' in review for e in [1, 2] for field in ['clarté', 'difficulté', 'unicité'])
rating_columns = [f'evaluateur_{e}_{field}' for e in [1, 2] for field in ['clarté', 'difficulté', 'unicité']]
filled = int(review[rating_columns].notna().all(axis=1).sum()) if human_ready else 0
print(f'Questions avec les six évaluations complètes : {filled} / {len(review)}')
if filled:
    complete = review[review[rating_columns].notna().all(axis=1)].copy()
    for field in ['clarté', 'difficulté', 'unicité']:
        kappa = cohen_kappa_from_review(complete, f'evaluateur_1_{field}', f'evaluateur_2_{field}', field)
        print(f"Kappa de Cohen ({field}) : {kappa:.3f}")
    for evaluator in [1, 2]:
        for field in ['clarté', 'difficulté', 'unicité']:
            review[f'evaluateur_{evaluator}_{field}'] = pd.to_numeric(review[f'evaluateur_{evaluator}_{field}'], errors='coerce')
    for field in ['clarté', 'difficulté', 'unicité']:
        review[f'moyenne_{field}'] = review[[f'evaluateur_1_{field}', f'evaluateur_2_{field}']].mean(axis=1)
    human_comparison = review.groupby('accepted', as_index=False).agg(
        nb_questions=('question_id', 'size'),
        clarté_moyenne=('moyenne_clarté', 'mean'),
        difficulté_moyenne=('moyenne_difficulté', 'mean'),
        unicité_moyenne=('moyenne_unicité', 'mean'))
    human_comparison['configuration'] = human_comparison['accepted'].map({True: 'Retenus par QA', False: 'Rejetés par QA'})
    human_comparison.to_csv(ROOT / 'results' / 'evaluation_humaine_comparative.csv', index=False, encoding='utf-8-sig')
    review.to_csv(review_path, index=False, encoding='utf-8-sig')
    display(human_comparison)
else:
    print('Grille humaine indisponible ; aucune annotation chargée.')

,mesure,valeur
0,QCM générés — sans filtre (baseline),120.000000
1,QCM conservés — QA local TF-IDF,113.000000
2,Taux de rejet QA local,0.058333
3,QCM dans la grille humaine,60.000000
4,QCM conservés — QA Gemini,103.000000
5,Taux de rejet QA Gemini,0.141667


,challenge_type,nombre,acceptes,taux_rejet
0,ambiguite_introduite,12,0,1.0
1,question_sans_reponse,12,0,1.0


Questions avec les six évaluations complètes : 60 / 60
Kappa de Cohen (clarté) : 0.050
Kappa de Cohen (difficulté) : 0.831
Kappa de Cohen (unicité) : 0.452


,accepted,nb_questions,clarté_moyenne,difficulté_moyenne,unicité_moyenne,configuration
0,False,7,1.714286,3.142857,1.714286,Rejetés par QA
1,True,53,2.556604,1.990566,2.962264,Retenus par QA


In [3]:
# Les notes ci-dessous proviennent des deux évaluatrices humaines du projet.
from scipy.stats import mannwhitneyu, spearmanr
from sklearn.metrics import cohen_kappa_score

human_review_path = PROCESSED / 'grille_evaluation_humaine.csv'
human_metrics = pd.DataFrame()
if human_review_path.exists():
    human_review = pd.read_csv(human_review_path)
    if len(human_review) != 60 or not human_review['question_id'].is_unique:
        raise ValueError('La grille humaine doit contenir 60 questions uniques.')
    rating_columns = [f'evaluateur_{e}_{criterion}' for e in [1, 2] for criterion in ['clarté', 'difficulté', 'unicité']]
    if not set(rating_columns).issubset(human_review.columns):
        raise ValueError('La grille doit contenir les six colonnes de notes des deux évaluatrices.')
    ratings = human_review[rating_columns].apply(pd.to_numeric, errors='raise')
    if ratings.isna().any(axis=None) or not ratings.apply(lambda column: column.between(1, 5)).all(axis=None):
        raise ValueError('Toutes les notes humaines doivent être renseignées sur une échelle de 1 à 5.')
    aligned = human_review.merge(
        candidates[['question_id', 'question', 'accepted']],
        on='question_id', how='left', suffixes=('_fichier', '_projet'), validate='one_to_one')
    if aligned['accepted_projet'].isna().any():
        raise ValueError('Certaines questions évaluées ne figurent pas dans les candidats du projet.')
    if not (aligned['question_fichier'] == aligned['question_projet']).all():
        raise ValueError('Les questions du fichier d’évaluation ne correspondent pas aux QCM actuels.')
    if not (aligned['accepted_fichier'].astype(bool) == aligned['accepted_projet'].astype(bool)).all():
        raise ValueError('Les décisions QA du fichier d’évaluation ne correspondent pas aux résultats actuels.')

    metric_rows = []
    for criterion in ['clarté', 'difficulté', 'unicité']:
        score_a = pd.to_numeric(aligned[f'evaluateur_1_{criterion}'], errors='raise')
        score_b = pd.to_numeric(aligned[f'evaluateur_2_{criterion}'], errors='raise')
        if score_a.isna().any() or score_b.isna().any() or not score_a.between(1, 5).all() or not score_b.between(1, 5).all():
            raise ValueError(f'Les notes pour {criterion} doivent toutes être renseignées entre 1 et 5.')
        row_mean = (score_a + score_b) / 2
        accepted_scores = row_mean[aligned['accepted_projet'].astype(bool)]
        rejected_scores = row_mean[~aligned['accepted_projet'].astype(bool)]
        test = mannwhitneyu(accepted_scores, rejected_scores, alternative='two-sided')
        metric_rows.append({
            'critere': criterion,
            'kappa_simple': cohen_kappa_score(score_a, score_b),
            'kappa_pondere_quadratique': cohen_kappa_score(score_a, score_b, weights='quadratic'),
            'spearman': spearmanr(score_a, score_b).statistic,
            'accord_exact': (score_a == score_b).mean(),
            'accord_a_un_point': ((score_a - score_b).abs() <= 1).mean(),
            'moyenne_retenues': accepted_scores.mean(),
            'moyenne_rejetees': rejected_scores.mean(),
            'p_mann_whitney': test.pvalue,
        })
    human_metrics = pd.DataFrame(metric_rows)
    human_metrics.to_csv(ROOT / 'results' / 'evaluation_humaine_metrics.csv', index=False, encoding='utf-8-sig')
    print('Évaluation par deux évaluatrices humaines.')
    print('Questions évaluées :', len(human_review), '| retenues :', int(aligned['accepted_projet'].sum()), '| rejetées :', int((~aligned['accepted_projet'].astype(bool)).sum()))
    display(human_metrics.round(3))
    report_lines = [
        '# Résultats de l’évaluation humaine des QCM',
        '',
        'Selon la confirmation du groupe, les notes ont été attribuées par deux évaluatrices humaines.',
        'Le fichier de synthèse reçu avec la grille portait une mention LLM contradictoire ; cette provenance a été corrigée dans le projet à la suite de cette clarification.',
        '',
        f"Échantillon : {len(human_review)} questions, dont {int(aligned['accepted_projet'].sum())} retenues et {int((~aligned['accepted_projet'].astype(bool)).sum())} rejetées par le filtre QA.",
        '',
        '| Critère | Kappa simple | Kappa quadratique | Spearman | Accord exact | Moy. retenues | Moy. rejetées | p (Mann–Whitney) |',
        '|---|---:|---:|---:|---:|---:|---:|---:|',
    ]
    for _, row in human_metrics.iterrows():
        report_lines.append(f"| {row['critere']} | {row['kappa_simple']:.3f} | {row['kappa_pondere_quadratique']:.3f} | {row['spearman']:.3f} | {row['accord_exact']:.1%} | {row['moyenne_retenues']:.3f} | {row['moyenne_rejetees']:.3f} | {row['p_mann_whitney']:.4f} |")
    report_lines += ['', 'Les tests entre QCM retenus et rejetés sont exploratoires : le groupe rejeté ne contient que sept questions.']
    report_path = ROOT / 'results' / 'evaluation_humaine_resume.md'
    report_path.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')
    print('Résumé enregistré :', report_path)
else:
    print('Grille humaine absente ; aucune note humaine chargée.')

Évaluation par deux évaluatrices humaines.
Questions évaluées : 60 | retenues : 53 | rejetées : 7


,critere,kappa_simple,kappa_pondere_quadratique,spearman,accord_exact,accord_a_un_point,moyenne_retenues,moyenne_rejetees,p_mann_whitney
0,clarté,0.050,0.559,0.784,0.350,1.0,2.557,1.714,0.006
1,difficulté,0.831,0.925,0.941,0.883,1.0,1.991,3.143,0.001
2,unicité,0.452,0.867,0.924,0.567,1.0,2.962,1.714,0.004


Résumé enregistré : D:\Generateur_QCM\results\evaluation_humaine_resume.md


## Export d’un article scientifique (version de travail)

Le document Word rassemble le contexte, la méthode et les résultats effectivement disponibles, y compris les notes des deux évaluatrices. Vérifier la mise en page finale dans Word ou LibreOffice : elle doit rester sous la limite de sept pages.

In [4]:
from docx import Document
from docx.shared import Pt, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH

doc = Document()
section = doc.sections[0]
section.top_margin = Cm(1.7)
section.bottom_margin = Cm(1.7)
section.left_margin = Cm(2.0)
section.right_margin = Cm(2.0)
normal = doc.styles['Normal']
normal.font.name = 'Calibri'
normal.font.size = Pt(9)
title = doc.add_heading('Génération de QCM à partir de supports de cours avec validation automatique par question-réponse', 0)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER
doc.add_paragraph('Projet de TALN — Sujet 11 | Filière IA & Science des Données, 5e année | EMSI Rabat | 2026–2027').alignment = WD_ALIGN_PARAGRAPH.CENTER
doc.add_heading('Résumé', level=1)
gemini_abstract = (f"Gemini a conservé {gemini_accepted_n} candidats sur {len(gemini_validated)}." if not gemini_validated.empty else 'La comparaison QA avec Gemini sera disponible après exécution du notebook 03 avec une clé API.')
doc.add_paragraph(f"Nous présentons une baseline de génération extractive de QCM à partir de six supports pédagogiques et comparons deux validateurs : un QA local fondé sur TF-IDF et Gemini 3.5 Flash-Lite, un modèle multilingue interrogé en français. Le corpus comprend {len(pd.read_csv(ROOT / 'data' / 'processed' / 'passages.csv')) if (ROOT / 'data' / 'processed' / 'passages.csv').exists() else 0} passages ; {baseline_n} candidats ont été produits, {advanced_n} retenus par le QA local (rejet : {rejection_rate:.1%}). {gemini_abstract} Deux évaluatrices humaines ont coté 60 QCM ; les résultats descriptifs et l’accord inter-évaluatrices sont rapportés.")
doc.add_paragraph('Mots-clés : génération de questions, QCM, question-réponse, TF-IDF, Gemini, français.')
doc.add_heading('1. Introduction', level=1)
doc.add_paragraph('La création de QCM à partir de supports pédagogiques est coûteuse et nécessite de vérifier que chaque question est répondable, que la réponse correcte est unique et que les distracteurs ne sont pas eux-mêmes corrects. Le sujet N11 demande une comparaison entre une base de référence et une approche avancée, une validation QA, l’analyse des rejets et une évaluation humaine. Notre objectif est de rendre ces étapes reproductibles sur six PDF fournis par l’enseignant.')
doc.add_heading('2. Données et prétraitement', level=1)
doc.add_paragraph('Le corpus est constitué de six supports fournis : vision par ordinateur (chapitres 1 et 2), employabilité (chapitres 1 et 2), fondamentaux MLOps, et DataFrames/Spark SQL. Les sources sont conservées dans data/sources. Le texte est extrait des PDF ; RapidOCR est utilisé localement pour les pages scannées. Les passages conservent le nom du document et la page. Les artefacts OCR, les formules et les tableaux constituent des sources d’erreur et requièrent une relecture.')
passage_frame = pd.read_csv(ROOT / 'data' / 'processed' / 'passages.csv')
source_summary = passage_frame.groupby('source_doc', as_index=False).agg(nb_passages=('passage_id', 'size'), nb_pages=('page', 'nunique'))
doc.add_paragraph(f"Taille du corpus : {len(passage_frame)} passages répartis sur {source_summary['nb_pages'].sum()} pages extraites.")
source_table = doc.add_table(rows=1, cols=3)
source_table.style = 'Light Shading Accent 1'
for cell, text in zip(source_table.rows[0].cells, ['Support', 'Pages', 'Passages']): cell.text = text
for _, source_row in source_summary.iterrows():
    cells = source_table.add_row().cells
    cells[0].text = str(source_row['source_doc'])
    cells[1].text = str(int(source_row['nb_pages']))
    cells[2].text = str(int(source_row['nb_passages']))
doc.add_heading('3. Méthodes', level=1)
doc.add_paragraph('La baseline sélectionne une phrase informative comme réponse, utilise les termes TF-IDF pour former une question à gabarit et choisit trois phrases distractrices. Le générateur est déterministe (graine 42), cite la source et produit quatre options. Cette méthode extractive facilite l’audit mais produit des questions parfois peu naturelles et des distracteurs de qualité variable.')
doc.add_paragraph('Le QA local TF-IDF sert de baseline de validation. En méthode avancée, Gemini 3.5 Flash-Lite reçoit par lots la question, ses quatre options et les extraits voisins du support, sans recevoir l’index de la réponse de référence. Il doit signaler si la question est répondable, sélectionner les options étayées et retourner une citation exacte. Le programme vérifie localement que la citation apparaît dans le contexte et compare l’option prédite à la référence ; les réponses non conformes ou ambiguës sont rejetées. Les appels API sont sans état (store=False), mais les extraits soumis transitent par Google. La comparaison et les contrôles d’ablation ne remplacent pas une validation humaine.')
doc.add_heading('4. Expériences et résultats', level=1)
gemini_result_text = (f"Sur les mêmes {len(gemini_validated)} candidats, Gemini en a conservé {gemini_accepted_n} (rejet : {gemini_rejection_rate:.1%}). " if not gemini_validated.empty else 'Les résultats Gemini ne sont pas encore disponibles ; exécuter le notebook 03 avec GEMINI_API_KEY. ')
doc.add_paragraph(f"Le corpus traité contient {baseline_n} questions candidates ; le QA local en a conservé {advanced_n} (rejet : {rejection_rate:.1%}). {gemini_result_text}Ces taux décrivent les décisions des validateurs, pas la qualité pédagogique réelle. L’évaluation humaine porte sur {len(review)} QCM issus des décisions locales ({int(review['accepted'].sum())} retenus et {int((~review['accepted'].astype(bool)).sum())} rejetés). Les contrôles et les sorties détaillées sont enregistrés dans results/.")
if control_path.exists():
    doc.add_paragraph('Contrôles d’ablation :')
    table = doc.add_table(rows=1, cols=3)
    table.style = 'Light Shading Accent 1'
    for cell, text in zip(table.rows[0].cells, ['Type', 'Nombre', 'Taux de rejet']): cell.text = text
    for _, row in pd.read_csv(control_path).iterrows():
        cells = table.add_row().cells
        cells[0].text = str(row['challenge_type'])
        cells[1].text = str(int(row['nombre']))
        cells[2].text = f"{row['taux_rejet']:.1%}"
if not human_metrics.empty:
    doc.add_heading('Évaluation humaine (n = 60)', level=2)
    doc.add_paragraph('Deux évaluatrices humaines ont coté la clarté, la difficulté et l’unicité des mêmes 60 questions. Les kappas simple et pondéré ainsi que les accords descriptifs sont calculés à partir de ces notes. Les différences entre QCM retenus et rejetés sont exploratoires, notamment en raison du petit nombre de questions rejetées.')
    human_table = doc.add_table(rows=1, cols=5)
    human_table.style = 'Light Shading Accent 1'
    for cell, text in zip(human_table.rows[0].cells, ['Critère', 'Kappa pondéré', 'Accord exact', 'Moy. retenues', 'Moy. rejetées']): cell.text = text
    for _, metric_row in human_metrics.iterrows():
        cells = human_table.add_row().cells
        cells[0].text = str(metric_row['critere'])
        cells[1].text = f"{metric_row['kappa_pondere_quadratique']:.2f}"
        cells[2].text = f"{metric_row['accord_exact']:.0%}"
        cells[3].text = f"{metric_row['moyenne_retenues']:.2f}"
        cells[4].text = f"{metric_row['moyenne_rejetees']:.2f}"
doc.add_heading('5. Analyse des erreurs et limites', level=1)
doc.add_paragraph('Les erreurs observées sont illustrées ci-dessous par des questions effectivement rejetées. Les motifs sont ceux du filtre automatique ; ils doivent être confirmés ou infirmés par les évaluateurs.')
validation_path = PROCESSED / 'qcm_valides_et_rejetes.csv'
validation_frame = pd.read_csv(validation_path)
rejected_examples = validation_frame.loc[~validation_frame['accepted']].head(4)
if len(rejected_examples):
    error_table = doc.add_table(rows=1, cols=3)
    error_table.style = 'Light Shading Accent 1'
    for cell, text in zip(error_table.rows[0].cells, ['Question', 'Preuve QA extraite', 'Motif de rejet']): cell.text = text
    for _, error in rejected_examples.iterrows():
        cells = error_table.add_row().cells
        cells[0].text = str(error['question'])[:220]
        cells[1].text = str(error['qa_evidence'])[:260]
        cells[2].text = str(error['rejection_reason'])
doc.add_paragraph('Les limites méthodologiques restent importantes : le générateur TF-IDF est extractif et à gabarits ; Gemini est un modèle génératif utilisé pour le QA, mais sa sortie n’est pas une confiance calibrée et peut contenir des erreurs. La vérification locale de citation ne garantit pas la validité sémantique. Les contrôles synthétiques ne démontrent pas la qualité pédagogique ; les résultats Gemini dépendent de la disponibilité, des quotas et des conditions de l’API. Une relecture humaine reste indispensable.')
doc.add_heading('6. Éthique, reproductibilité et conclusion', level=1)
doc.add_paragraph('Les supports sont conservés localement et les extraits envoyés à Gemini sont limités au contexte nécessaire à la validation ; l’API est appelée avec store=False. La clé reste dans le fichier local .env, ignoré par Git, et n’est pas enregistrée dans le notebook. Les passages citent leur document et leur page afin de faciliter la vérification. La graine aléatoire est fixée à 42 et les dépendances sont listées dans requirements.txt. Le système est un prototype d’assistance à l’enseignant et ne doit pas publier de QCM sans relecture humaine.')
if filled == len(review):
    doc.add_paragraph(f"Évaluation humaine : deux évaluatrices ont complété les trois critères pour les 60 questions. La clarté moyenne est de {human_metrics.loc[human_metrics['critere'] == 'clarté', 'moyenne_retenues'].iloc[0]:.2f}/5 pour les QCM retenus contre {human_metrics.loc[human_metrics['critere'] == 'clarté', 'moyenne_rejetees'].iloc[0]:.2f}/5 pour les rejetés. Les résultats détaillés et les kappas de Cohen figurent dans results/evaluation_humaine_metrics.csv.")
else:
    doc.add_paragraph(f"Évaluation humaine inachevée : {filled} question(s) sur {len(review)} disposent des six notes ; la grille doit être complétée par deux évaluatrices avant la remise.")
doc.add_heading('Références', level=1)
doc.add_paragraph('Cohen, J. (1960). A coefficient of agreement for nominal scales. Educational and Psychological Measurement, 20(1), 37–46. Cohen, J. (1968). Weighted kappa: Nominal scale agreement with provision for scaled disagreement. Psychological Bulletin, 70(4), 213–220. Google. (2026). Gemini API documentation: Interactions API and structured outputs. https://aistudio.google.com/docs. Salton, G., Wong, A., & Yang, C. S. (1975). A vector space model for automatic indexing. Communications of the ACM, 18(11), 613–620. Supports pédagogiques fournis par l’enseignant (six PDF listés dans data/sources). Cahier des charges du projet de TALN, Sujet 11, EMSI Rabat, année universitaire 2026–2027.')
article_path = ROOT / 'article_projet.docx'
doc.save(article_path)
print(f'Article de travail enregistré : {article_path}')
print('Avant remise : vérifier la pagination (maximum 7 pages), harmoniser les références et relire les décisions du validateur Gemini.')

Article de travail enregistré : D:\Generateur_QCM\article_projet.docx
Avant remise : vérifier la pagination (maximum 7 pages), harmoniser les références et relire les décisions du validateur Gemini.
